# Malaria detection from blood-smear cell images


Malaria is diagnosed by looking at blood smears under a microscope, which takes trained staff and time. This notebook trains two convolutional neural networks (CNNs) to tell parasitized (infected) red blood cells from uninfected ones, using about 27,500 labelled cell images from the U.S. National Library of Medicine.

1. A small CNN trained from scratch (baseline)
2. MobileNetV2 pretrained on ImageNet, then fine-tuned (transfer learning)

Then it checks how many infected cells each model misses and uses Grad-CAM to show where the model looked.


> Educational project. These models are not diagnostic tools.

## 1. Setup

In [1]:
import os, sys

# On Colab, clone the repo so the src/ code is available. Change to your GitHub username.
REPO_URL = "https://github.com/moraraba/malaria-detection-cnn.git"

if "google.colab" in sys.modules and not os.path.exists("src"):
    !git clone -q {REPO_URL} repo
    %cd repo

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from IPython.display import Image, display

from src.data import CLASS_NAMES, load_splits
from src.models import build_baseline_cnn, build_transfer_model, compile_model, unfreeze_top
from src.evaluate import evaluate, predict
from src.gradcam import save_grid

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "none (training will be slow)")

In [ ]:
# Settings
DATA = "tfds"          # or a folder holding Parasitized/ and Uninfected/ (the Kaggle download)
WEIGHTS = "imagenet"   # pretrained weights for MobileNetV2
IMG_SIZE = 128
BATCH_SIZE = 64
BASELINE_EPOCHS = 15
TRANSFER_EPOCHS = 10
FINE_TUNE_EPOCHS = 5
SEED = 42
OUT = "outputs"

keras.utils.set_random_seed(SEED)
os.makedirs(f"{OUT}/figures", exist_ok=True)

## 2. Load the data

The images are split once, the same way every time: 70% to train, 15% to tune training (validation), and 15% held back for the final test. Pixel values stay 0–255; each model rescales its own input.

In [ ]:
train_ds, val_ds, test_ds, sizes = load_splits(DATA, IMG_SIZE, BATCH_SIZE, SEED)
print({k: f"{v:,}" for k, v in sizes.items()})

In [ ]:
# Class balance in the training set
labels = np.concatenate([y.numpy() for _, y in train_ds])
counts = pd.Series(labels).map(dict(enumerate(CLASS_NAMES))).value_counts()
print(counts.to_string())

In [ ]:
# A look at the cells: infected cells show purple-stained parasites
images, labels = next(iter(train_ds))
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for ax, img, lab in zip(axes.flat, images, labels):
    ax.imshow(img.numpy().astype("uint8"))
    ax.set_title(CLASS_NAMES[int(lab)], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Baseline: a small CNN from scratch

Three convolution blocks learn edges, then shapes, then parasite-like blobs. Random flips, rotations and zooms (data augmentation) stop the model memorising individual images. Early stopping keeps the epoch with the lowest validation loss.

In [ ]:
def callbacks(path):
    return [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(path, monitor="val_loss", save_best_only=True),
    ]

baseline = compile_model(build_baseline_cnn(IMG_SIZE), 1e-3)
baseline.summary()
hist_base = baseline.fit(train_ds, validation_data=val_ds, epochs=BASELINE_EPOCHS,
                         callbacks=callbacks(f"{OUT}/baseline.keras")).history

## 4. Transfer learning: MobileNetV2

MobileNetV2 has already learned general visual features from 1.4 million ImageNet photos. We freeze it, train only a new final layer, then unfreeze its top blocks and fine-tune them with a learning rate 100 times smaller, so the pretrained features are adjusted rather than overwritten.

In [ ]:
transfer = compile_model(build_transfer_model(IMG_SIZE, weights=WEIGHTS), 1e-3)
hist_tr = transfer.fit(train_ds, validation_data=val_ds, epochs=TRANSFER_EPOCHS,
                       callbacks=callbacks(f"{OUT}/mobilenet.keras")).history

In [ ]:
# Fine-tune the top blocks
start = len(hist_tr["loss"])
unfreeze_top(transfer)
compile_model(transfer, 1e-5)
ft_callbacks = callbacks(f"{OUT}/mobilenet.keras")
ft_callbacks[1] = keras.callbacks.ModelCheckpoint(
    f"{OUT}/mobilenet.keras", monitor="val_loss", save_best_only=True,
    initial_value_threshold=min(hist_tr["val_loss"]))
hist_ft = transfer.fit(train_ds, validation_data=val_ds, initial_epoch=start,
                       epochs=start + FINE_TUNE_EPOCHS, callbacks=ft_callbacks).history
for k in hist_tr:
    hist_tr[k] += hist_ft.get(k, [])

# Keep the best model from either phase
transfer = keras.models.load_model(f"{OUT}/mobilenet.keras")

In [ ]:
# Training curves: a widening gap between train and validation means overfitting
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, (name, h) in zip(axes, [("Baseline CNN", hist_base), ("MobileNetV2", hist_tr)]):
    ax.plot(h["accuracy"], label="train")
    ax.plot(h["val_accuracy"], label="validation")
    if name == "MobileNetV2":
        ax.axvline(start - 0.5, color="grey", ls="--", lw=1)
        ax.text(start - 0.4, ax.get_ylim()[0], " fine-tune", color="grey", fontsize=8, va="bottom")
    ax.set_title(name); ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy"); ax.legend()
plt.tight_layout()
plt.show()

## 5. Test-set results

Accuracy alone hides the costly mistake. **Sensitivity** (recall) is the share of infected cells caught; a missed infection is worse than a false alarm, which a technician can re-check.

In [ ]:
baseline = keras.models.load_model(f"{OUT}/baseline.keras")
results = [evaluate(baseline, test_ds, OUT, "baseline"), evaluate(transfer, test_ds, OUT, "mobilenet")]
table = pd.DataFrame(results).set_index("model")[
    ["accuracy", "sensitivity_recall", "specificity", "precision", "f1", "roc_auc",
     "missed_infected_cells", "false_alarms"]]
table.round(3)

In [ ]:
for name in ["baseline", "mobilenet"]:
    display(Image(f"{OUT}/figures/{name}_confusion_matrix.png", width=380))
display(Image(f"{OUT}/figures/mobilenet_roc.png", width=380))

### Trading false alarms for fewer missed infections

The model outputs a probability. Lowering the cut-off from 0.5 flags more cells as infected: fewer missed infections, more false alarms.

In [ ]:
y_true, y_prob = predict(transfer, test_ds)
rows = []
for t in [0.2, 0.3, 0.4, 0.5, 0.6]:
    pred = y_prob >= t
    tp = int((pred & (y_true == 1)).sum()); fn = int((~pred & (y_true == 1)).sum())
    fp = int((pred & (y_true == 0)).sum()); tn = int((~pred & (y_true == 0)).sum())
    rows.append({"threshold": t, "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp),
                 "missed_infected_cells": fn, "false_alarms": fp})
pd.DataFrame(rows).set_index("threshold").round(3)

## 6. Where did the model look? (Grad-CAM)

Grad-CAM colours the parts of the image that pushed the prediction most (red = strongest). For a trustworthy model, heat on infected cells should sit on the purple parasite, not on the background or the image border.

In [ ]:
save_grid(transfer, test_ds, f"{OUT}/figures/mobilenet_gradcam.png", n=8)
display(Image(f"{OUT}/figures/mobilenet_gradcam.png"))

## 7. Save results

Copy `outputs/` (models, metrics, figures) back into the repo, or download it from Colab's file panel. Then fill in the results table in the README.

In [ ]:
for r in results:
    json.dump(r, open(f"{OUT}/{r['model']}_metrics.json", "w"), indent=2)
print(table.round(3).to_markdown())

## Limitations

* **Same patient in train and test.** The images are split by cell, not by patient, and each patient contributed many cells. Cells from one person can land in both sets, which flatters test accuracy. The public release doesn't include patient IDs, so this split can't be fixed here.
* **Label noise.** Published reviews of this dataset have found some mislabelled cells.
* **One lab, one stain, one microscope.** Images from other clinics, phones or stains may look different, and accuracy could drop.
* **Cells, not patients.** A diagnosis needs a whole smear: cell segmentation, counting, and parasite species. This project covers only the single-cell step.